# Complexity-08d — Indépendance prouvée : le Castor affairé, une borne de couverture, et la preuve comme objet

*Versant formel du fil des machines — distillation des trois résultats `jamesyc` d'octobre 2026 — See [#27477](https://github.com/jsboige/CoursIA/issues/27477) (jalon 1), [#20607](https://github.com/jsboige/CoursIA/issues/20607) (EPIC Origami Complexity).*

**Navigation** : [<< Complexity-08c — Machines de Gödel](Complexity-08c-MachinesDeGoedel-Python.ipynb) | [Index de la série](README.md) | [Complexity-08e — Machines à compte à rebours →](Complexity-08e-NF-MachinesCompteARebours-Python.ipynb)
**Navigation** : [→ Explorer la version Rust](Complexity-08d-IndependanceProuvee-BusyBeaver-Covering-Rust.ipynb)

Ce carnet poursuit le fil des grands nombres ([`Complexity-07b`](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb), [`Complexity-07c`](Complexity-07c-LeBilanMathocalypse-Python.ipynb)) et des machines ([`Complexity-08c`](Complexity-08c-MachinesDeGoedel-Python.ipynb)) par son **versant formel** : non plus *décrire* des machines de Turing ou des bornes, mais **prouver en Lean** un résultat de frontière — et lire ce que la preuve établit exactement.

Trois résultats publiés en octobre 2026 (dépôts `jamesyc/*`, tous déjà formalisés et reproductibles) servent de fil :

| Résultat | Énoncé | Formalisation | Licence |
|---|---|---|---|
| **PA196** | une machine de Turing à 196 états dont PA ne prouve ni l'arrêt ni la non-arrêt | Lean 4 (`jamesyc/BusyBeaver`, PA196) | MIT |
| **C(24,14,4) ≥ 20** | borne inférieure d'un nombre de couverture (record, antérieur 19) ; C(25,15,5) ≥ 34 par Schönheim | Lean 4 (`jamesyc/covering`) | Apache-2.0 |
| **κ = 83/10¹²** | gain conditionnel pour la multiplication d'entiers (recherche, **non revu**) | prose + certificat (`jamesyc/integer-mult-bounds`) | Apache-2.0 |

**Ce que ce carnet NE fait PAS** : il n'importe aucun lac Lean (le gap 4.35.0 vs notre 4.33.1 relève du jalon 2 de [#27477](https://github.com/jsboige/CoursIA/issues/27477)). Il **présente** les résultats depuis leurs papiers archivés, vérifie au passage les affirmations *arithmétiques et combinatoires* qui ne demandent aucun moteur externe (Schönheim, décompte d'incidences, la borne 2⁻³⁴), et distingue soigneusement le **théorème vérifié** du **résultat conditionnel**.

## 0. Trois objets, trois statuts de preuve

Avant tout énoncé, la distinction qui gouverne le carnet :

1. **Théorème formalisé et audité** (PA196, C(24,14,4)) : la preuve existe en Lean, un *build* passe, et un **audit d'axiomes** borne ce en quoi elle croit. Pour C(24,14,4), l'audit publié ne retient que `propext`, `Classical.choice`, `Quot.sound` — **aucun** `sorryAx`, aucun `native_decide` : la borne est une conséquence de la logique de Lean avec le choix classique, rien d'autre.
2. **Résultat conditionnel** (κ pour la multiplication d'entiers) : l'auteur écrit lui-même « *research draft awaiting independent review* » et épingle les interfaces amont « *remain assumptions* ». On le présente comme tel — jamais comme un théorème.

Cette gradation *est* le contenu pédagogique : une preuve formelle n'efface pas la question « de quoi dépend-elle ? », elle la rend **répondable mécaniquement**.

## 1. PA196 — une machine finie qui échappe à Peano

Le théorème (Chang, *Nonhalting and PA-independence for a 196-state Turing machine*) :

> Il existe une machine de Turing à deux symboles, **196 états utiles**, qui **ne s'arrête jamais** sur ruban vierge ; et si PA est consistant, **PA ne prouve ni sa phrase d'arrêt ni sa négation**.

La structure Lean (lue dans `PA196/docs/result.md`) est un chaînage court et net :

- `PA196.Literal.working_count` / `row_count` — la machine littérale a 196 états ; le compte de lignes de transition est rendu par ces champs ;
- `PA196.Startup.startup_exact` — le calcul sur ruban vierge atteint une configuration de départ propre après **58 154 pas** ;
- `PA196.machine_nonhalting` — la machine ne s'arrête jamais depuis le ruban vierge ;
- `PA196.Arithmetic.nonhalting_true` — les entiers standard satisfont la négation `N` de la phrase d'arrêt ;
- `PA196.Arithmetic.pa_proves_nonhalting_consistency` — **PA prouve `N → Con(PA)`** ;
- `PA196.Arithmetic.two_sided_independence` — **PA ne prouve ni `N` ni `H`**.

Le pivot logique est le **second théorème d'incomplétude de Gödel** : si PA prouvait `N` (la non-arrêt), il prouverait sa propre consistance `Con(PA)` (par la clause `N → Con(PA)`), ce que Gödel interdit à une théorie consistante. La machine *code* une recherche de contradiction dans PA ; sa non-arrêt **équivaut**, en interne, à la consistance de PA.

In [1]:
# Le pivot de Gödel, verifie comme une implication pure (aucun moteur externe).
# Si PA est consistant et que PA prouve N -> Con(PA) (clause du papier),
# alors PA prouver N entrainerait PA prouve Con(PA) -- interdit par Goedel-2.
# On materialise la clause sur un modele booleen a trois atomes : N, ConPA, PA_proves.
# Ici on verifie la SEULE chose calculable sans moteur : la validite de l'enchainement.

def goedel_pivot(pa_proves_N: bool, pa_proves_N_impl_Con: bool) -> str:
    """Retourne le verdict de coherence sous Goedel-2 (PA supposee consistante)."""
    if pa_proves_N and pa_proves_N_impl_Con:
        return "PA prouverait Con(PA) -> contradiction avec Goedel-2 (PA consistante). Donc PA ne prouve pas N."
    return "Configuration coherente avec l'independance."

# La clause N -> Con(PA) est un theoreme de PA (prouvee dans le papier) : pa_proves_N_impl_Con = True.
print("Cas PA prouve N      :", goedel_pivot(True, True))
print("Cas PA ne prouve pas N:", goedel_pivot(False, True))
print()
print("Lecture : la seconde ligne est le regime reel. PA196 exhibe une machine PRECISE")
print("dont la non-arret N est vraie sur les entiers standard mais non prouvable dans PA :")
print("c'est un témoin fini d'independance, pas une phrase de Goedel abstraite.")

Cas PA prouve N      : PA prouverait Con(PA) -> contradiction avec Goedel-2 (PA consistante). Donc PA ne prouve pas N.
Cas PA ne prouve pas N: Configuration coherente avec l'independance.

Lecture : la seconde ligne est le regime reel. PA196 exhibe une machine PRECISE
dont la non-arret N est vraie sur les entiers standard mais non prouvable dans PA :
c'est un témoin fini d'independance, pas une phrase de Goedel abstraite.


### Lecture du résultat

Le calcul ci-dessus ne prouve rien sur PA — il **rend explicite** l'enchaînement que la preuve Lean établit, elle, rigoureusement : la clause `N → Con(PA)` est la charnière, et Gödel-2 fait le reste. Ce que PA196 apporte de neuf par rapport à un énoncé d'indépendance « abstrait » (Yedidia–Aaronson, la lignée BBChallenge), c'est une **machine concrète de 196 états**, à la frontière actuelle des machines dont l'indépendance d'un système formel est *prouvée*. Le carnet ne re-démontre pas ce fait : il le cite depuis sa preuve formalisée, et isole le point logique qu'un lecteur peut vérifier à la main.

## 2. C(24,14,4) ≥ 20 — la borne de couverture, et pourquoi 19 était accessible

Un **covering design** `C(v, k, t)` est le nombre minimal de blocs de taille `k` sur `v` points tel que **tout** sous-ensemble de `t` points soit contenu dans au moins un bloc. Le résultat (Chang, *A structural lower bound for C(24,14,4)*) :

> **C(24,14,4) ≥ 20** (borne antérieure 19), et par la borne de Schönheim **C(25,15,5) ≥ 34** (antérieure 32).

L'argument est **structurel**, non une recherche exhaustive : on *suppose* un covering à 19 blocs et on dérive une contradiction. Le décompte d'incidences ci-dessous est recalculé exactement. Il emploie toutefois un degré minimal de 11, qui n'est **pas** établi par le comptage local : il est importé du résultat sharp `C(23,13,3) = 11` du papier de Chang.

Cette borne est desormais **branchee** : le carnet [`Lean-43-Borne-Couverture-C24-14-4-Lean`](../../SymbolicAI/Lean/Lean-43-Borne-Couverture-C24-14-4-Lean.ipynb) consomme le paquet `covering` de l'amont **tel quel**, au commit `e28962cf...`, par la clause Lake `from git ... @ "<sha>"` du lac [`covering_lean/`](../../SymbolicAI/Lean/covering_lean/README.md) — aucun fork, aucune copie. Le build passe sous l'organe borne `lean_exec.py`, et l'audit `#print axioms` des deux theoremes ne retient que `propext`, `Classical.choice`, `Quot.sound`, **concordant** avec la liste que l'amont publie lui-meme dans son `comparator.json`. Le present carnet en donne la lecture ; le carnet Lean-43 en donne l'objet.

In [2]:
# Decompte d'incidences d'un hypothétique (24,14,4)-covering a 19 blocs.
from math import ceil, comb

v, k, t, b = 24, 14, 4, 19

# Tout point appartient a assez de blocs pour couvrir les triples qui le contiennent.
deg_comptage = ceil(comb(v - 1, t - 1) / comb(k - 1, t - 1))
print(f"Comptage local : ceil(C(23,3) / C(13,3)) = ceil(1771 / 286) = {deg_comptage}.")
print("Ce comptage seul ne donne pas le degre 11.")

# Les blocs contenant un point fixe se derivent en un (23,13,3)-covering.
# Chang etablit C(23,13,3) = 11 : ce resultat importe donne le degre minimal.
deg_min = 11
print(f"Resultat importe de Chang : C(23,13,3) = {deg_min} ; deg_min = {deg_min}.")

# Incidences fournies par 19 blocs de taille 14 :
incidences = b * k
# Incidences requises si les 24 points ont chacun degre >= 11 :
requis = v * deg_min
exces = incidences - requis
print(f"{b} blocs x {k} points = {incidences} incidences ; requis >= {requis} ; EXCES = {exces}")

# Donc au moins (24 - exces) points ont degre EXACTEMENT 11 :
points_deg_exact = v - exces
print(f"Au moins {points_deg_exact} points ont degre exactement {deg_min}.")
print("Leurs vecteurs d'incidence vivent dans R^19 :")
print(f"  {points_deg_exact} vecteurs dans R^{b} -> au moins {points_deg_exact - b} relations lineaires independantes.")

# Borne de Schonheim : C(25,15,5) >= ceil(25 * C(24,14,4) / 15).
lower_24 = 20  # le resultat prouve
sch = ceil(25 * lower_24 / 15)
print()
print(f"Schonheim : C(25,15,5) >= ceil(25 * {lower_24} / 15) = ceil({25*lower_24/15:.3f}) = {sch}")
print("(Le README confirme : 'ceil(25 * 20 / 15) = 34'.)")

Comptage local : ceil(C(23,3) / C(13,3)) = ceil(1771 / 286) = 7.
Ce comptage seul ne donne pas le degre 11.
Resultat importe de Chang : C(23,13,3) = 11 ; deg_min = 11.
19 blocs x 14 points = 266 incidences ; requis >= 264 ; EXCES = 2
Au moins 22 points ont degre exactement 11.
Leurs vecteurs d'incidence vivent dans R^19 :
  22 vecteurs dans R^19 -> au moins 3 relations lineaires independantes.

Schonheim : C(25,15,5) >= ceil(25 * 20 / 15) = ceil(33.333) = 34
(Le README confirme : 'ceil(25 * 20 / 15) = 34'.)


### Lecture du résultat

Le décompte est le ressort : 19 blocs ne fournissent que **2 incidences d'excès** sur les 264 requises — c'est *parce que* la marge est aussi étroite que le cas d'égalité est analysable. Au moins 22 points ont degré exactement 11 ; leurs vecteurs d'incidence (dans ℝ¹⁹) portent au moins **3 relations linéaires indépendantes**, qu'une **identité somme-de-carrés** force à se répartir en groupes disjoints « équilibrés » sur ≤ 16 points. Le reste de la preuve exclut ces groupes par rigidité (le théorème 2.1 : onze blocs de taille 12 couvrant les triples de 22 points partitionnent en **onze paires de jumeaux**, un 2-(11,6,3) design symétrique après compression). On aboutit à deux contradictions de comptage élémentaires.

La borne C(25,15,5) ≥ 34 n'est **pas** une seconde preuve : c'est Schönheim appliqué au résultat, ⌈25·20/15⌉ = 34, confirmé par le calcul ci-dessus. La formalisation Lean publie les deux théorèmes (`Covering.Palomar.covering_24_14_4_lower_bound`, `covering_25_15_5_lower_bound`), audités sans `sorryAx`.

## 3. κ = 83/10¹² — le résultat conditionnel, traité comme tel

Le troisième objet est d'une autre nature. La note de Colkitt (*Compact dirty controls and a conditional integer-multiplication saving*) annonce un témoin **κ = 83/10¹² > 2⁻³⁴** pour la multiplication d'entiers sous `n log n`, en remplaçant des permutations pleines par des échanges de champs de contrôle compacts. Mais l'auteur écrit explicitement que les interfaces algorithmiques et analytiques du manuscrit amont « *remain assumptions* » et que c'est un « *research draft awaiting independent review* ».

On le cite donc **sans aucune revendication de théorème** : c'est un certificat de recherche, pas un résultat établi. Vérifions seulement la cohérence arithmétique de la borne comparée.

In [3]:
# Coherence arithmetique de la borne comparee (rien de plus n'est revendique).
kappa = 83 / 10**12
two_pow = 2.0**-34
print(f"kappa = 83/10^12 = {kappa:.3e}")
print(f"2^-34           = {two_pow:.3e}")
print(f"kappa > 2^-34 ? {kappa > two_pow}")
print(f"Rapport kappa / 2^-34 = {kappa / two_pow:.2f}")
print()
print("Rappel : ce calcul n'etablit PAS le gain de multiplication d'entiers.")
print("Il verifie uniquement que la constante annoncee depasse la reference 2^-34,")
print("comme le papier l'affirme. Le resultat reste CONDITIONNEL et NON REVU.")

kappa = 83/10^12 = 8.300e-11
2^-34           = 5.821e-11
kappa > 2^-34 ? True
Rapport kappa / 2^-34 = 1.43

Rappel : ce calcul n'etablit PAS le gain de multiplication d'entiers.
Il verifie uniquement que la constante annoncee depasse la reference 2^-34,
comme le papier l'affirme. Le resultat reste CONDITIONNEL et NON REVU.


### Lecture du résultat

κ = 8,3·10⁻¹¹ dépasse bien 2⁻³⁴ ≈ 5,8·10⁻¹¹ (rapport ≈ 1,43) : l'affirmation *numérique* du papier est cohérente. C'est tout ce qu'on peut dire sans moteur externe ni revue. Le contraste avec les sections 1 et 2 est la leçon : **le même carnet peut héberger un théorème audité et une piste conditionnelle, à condition de ne jamais les présenter au même rang.** La bibliographie distingue les deux ; le corps du texte aussi.

## 4. Méthodologie — une campagne multi-agents, lue comme notre propre pratique

Le dépôt `covering` documente comment le résultat a été obtenu : **sept agents** (un coordinateur, six sous-agents de recherche « Astra ») sur une plateforme de recherche, les 1–3 octobre 2026. L'auteur a choisi le problème, dirigé, questionné. Ce qui y ressemble à notre propre discipline de flotte vaut d'être nommé :

- **Plausible ≠ vérifié.** Le *build* réussi d'un agent est suivi **séparément** d'un *rebuild indépendant*. C'est exactement notre règle « le producteur n'est jamais son propre vérificateur ».
- **La review porte sur l'énoncé, pas seulement la preuve** : il faut vérifier que l'énoncé formel décrit *bien* un covering design, et que les lemmes intermédiaires portent sur les mêmes blocs et points que le théorème final — notre « lire l'état d'une review n'est pas lire la review », transposé au formalisme.
- **Pool de tâches partagé** : après la preuve, l'équipe est passée d'assignations fixes à un **pool partagé** où un agent propose une tâche, le coordinateur la scope, un autre la prend. Notre tapis + claims, mutatis mutandis.

Le résultat C(24,14,4) a émergé comme **lemme intermédiaire** d'une route de borne inférieure — un rappel que les livrables utiles naissent souvent en chemin, pas au terme visé. La campagne continue vers la possibilité d'un covering à 20 blocs ; le dépôt, lui, établit des **bornes**, sans déterminer les nombres de couverture ni résoudre la question amont à 41 blocs.

## 5. Exercices

Trois exercices, stubs compilables sans erreur volontaire (C.1) : le carnet s'exécute de bout en bout même non complétés.

### Exercice 1 — Le pivot de Gödel, par l'autre bout
La clause `N → Con(PA)` fait tout le travail. Que se passerait-il si une machine codait une théorie **plus faible** que PA (par exemple Robinson Q), pour laquelle Gödel-2 s'applique aussi ? Esquisser pourquoi l'indépendance se transfère, et ce qui pourrait la bloquer.

In [4]:
# Exercice 1 -- le pivot de Goedel sur une theorie plus faible
# Etape 1 : rappeler la condition d'application de Goedel-2 (quelle force minimale ?).
# Etape 2 : dire si la clause N -> Con(T) se prouve dans T pour T = Q (Robinson).
# Etape 3 : conclure sur le transfert d'independance, ou nommer le point qui coince.
# TODO etudiant
print("Exercice a completer : Goedel-2 exige une theorie qui represente assez d'arithmetique ;")
print("discuter si Q suffit, et si la simulation N -> Con(Q) reste interne a Q.")

Exercice a completer : Goedel-2 exige une theorie qui represente assez d'arithmetique ;
discuter si Q suffit, et si la simulation N -> Con(Q) reste interne a Q.


### Exercice 2 — La marge d'incidences
Le ressort de C(24,14,4) est l'excès de 2 incidences (266 contre 264). Généraliser : pour un `(v,k,4)` covering à `b` blocs où chaque point a degré ≥ `d`, exprimer l'excès `b·k − v·d`, et trouver d'autres triplets où cet excès est assez petit pour rendre le cas d'égalité analysable.

In [5]:
# Exercice 2 -- generaliser le decompte d'incidences
# Etape 1 : ecrire exces(v, k, b, d) = b*k - v*d.
# Etape 2 : balayer quelques (v, k, b, d) plausibles et reperer exces petit (0..4).
# Etape 3 : commenter pourquoi un petit exces ouvre une analyse structurelle.
# TODO etudiant
# def exces(v, k, b, d): return b*k - v*d
print("Exercice a completer : exces(v,k,b,d) = b*k - v*d ; chercher les triplets a exces 0..4.")

Exercice a completer : exces(v,k,b,d) = b*k - v*d ; chercher les triplets a exces 0..4.


### Exercice 3 — Classer un résultat avant de le citer
On vous donne trois affirmations trouvées en ligne : (a) un théorème Lean dont l'audit ne liste que `propext`/`Classical.choice`/`Quot.sound` ; (b) un préprint « conditionnel, en attente de revue » ; (c) une capture d'écran d'un nombre sans source. Pour chacune, écrire la phrase de citation honnête que vous mettriez dans un carnet, et ce qu'il faudrait vérifier en premier.

In [6]:
# Exercice 3 -- gradation des preuves avant citation
# Etape 1 : pour (a), nommer ce que l'audit d'axiomes garantit ET ce qu'il ne garantit pas.
# Etape 2 : pour (b), ecrire la formulation qui ne revendique pas un theoreme.
# Etape 3 : pour (c), dire pourquoi on ne le cite PAS tant que la source n'est pas archivee.
# TODO etudiant
print("Exercice a completer : (a) audite = theoreme modulo les axiomes listes ;")
print("(b) conditionnel = le dire, ne pas conclure ; (c) pas de source = pas de citation.")

Exercice a completer : (a) audite = theoreme modulo les axiomes listes ;
(b) conditionnel = le dire, ne pas conclure ; (c) pas de source = pas de citation.


## 6. Conclusion et ponts

Trois objets, trois statuts, une seule exigence : **dire exactement ce qui est prouvé, et de quoi cela dépend.** PA196 montre qu'une machine finie peut échapper à PA par Gödel-2 ; C(24,14,4) ≥ 20 montre une borne combinatoire devenue *théorème audité* ; κ montre qu'on peut présenter une piste conditionnelle sans la trahir.

Ponts dans le dépôt : le fil des grands nombres ([`Complexity-07b`](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb)) décrit le Castor affairé du côté descriptif ; les machines ([`Complexity-08c`](Complexity-08c-MachinesDeGoedel-Python.ipynb)) côtoient l'auto-référence ; l'EPIC Origami Complexity ([#20607](https://github.com/jsboige/CoursIA/issues/20607)) porte la maison. La **suite** (jalons 2–3 de [#27477](https://github.com/jsboige/CoursIA/issues/27477)) etait de **brancher** les preuves Lean elles-mêmes : c'est fait. Le carnet [`Lean-42`](../../SymbolicAI/Lean/Lean-42-Independance-Arithmetique-BusyBeaver-Lean.ipynb) branche PA196, le carnet [`Lean-43`](../../SymbolicAI/Lean/Lean-43-Borne-Couverture-C24-14-4-Lean.ipynb) branche C(24,14,4) ≥ 20 — tous deux en **CONSUMER_PINNED** : le paquet amont est consomme tel quel, au commit epingle, dans un lac dedie dont le toolchain est celui de l'amont. Ce carnet les a presentees ; les deux carnets Lean les importent.

## 7. Références

Sources primaires — toutes **archivées au gisement** avant rédaction (bibliography-hygiene), identité vérifiée sur la première page, et c'est depuis ces copies que le carnet est écrit :

- **Chang, J. (2026).** *Nonhalting and PA-independence for a 196-state Turing machine.* Dépôt `jamesyc/BusyBeaver` (licence **MIT**, formalisation Lean 4 PA196). Archivé — `G:\\Mon Drive\\MyIA\\IA\\Bibliographie IA\\Automata\\Complexity Theory and Hardness\\2026 - Chang - Nonhalting and PA-independence for a 196-state Turing machine (github jamesyc-BusyBeaver, MIT, Lean formalise).pdf`. Source des énoncés de la section 1 et du chaînage `result.md` (196 états, 58 154 pas, clause `N → Con(PA)`, `two_sided_independence`).
- **Chang, J. (2026).** *A structural lower bound for C(24,14,4)* (11 octobre 2026). Dépôt `jamesyc/covering` (licence **Apache-2.0**, formalisation Lean 4 publiée Palomar). Archivé — `G:\\Mon Drive\\MyIA\\IA\\Bibliographie IA\\Automata\\Complexity Theory and Hardness\\2026 - Chang - A structural lower bound for C(24,14,4) (github jamesyc-covering, Apache-2.0, Lean formalise).pdf`. Source de la section 2 (excès 2 incidences, ≥ 22 points de degré 11, ≥ 3 relations, théorème de rigidité 2.1, Schönheim ⌈25·20/15⌉ = 34) et de la section 4 (campagne 7 agents, plausible ≠ vérifié, pool partagé). Audit d'axiomes : `propext`, `Classical.choice`, `Quot.sound`.
- **Colkitt, D. (2026).** *Compact dirty controls and a conditional integer-multiplication saving* (7 octobre 2026). Dépôt `jamesyc/integer-mult-bounds` (licence **Apache-2.0**). Archivé — `G:\\Mon Drive\\MyIA\\IA\\Bibliographie IA\\Automata\\Complexity Theory and Hardness\\2026 - Colkitt - Compact dirty controls and a conditional integer-multiplication saving (github jamesyc-integer-mult-bounds, conditionnel non-revu).pdf`. Source de la section 3 (κ = 83/10¹² > 2⁻³⁴). **Conditionnel, en attente de revue indépendante** — présenté comme tel.

Contexte (non cité dans le corps, pour le lecteur) : Yedidia & Aaronson (indépendance BB vs ZFC), la lignée Rado / BBChallenge ; la borne de Schönheim pour les covering designs.